In [6]:
# 로더/경로/모델/옵티마이저
from preparation import get_train_val_loaders, get_model_paths
from tqdm.auto import tqdm

import torch
from torchvision.models.detection.ssd import ssd300_vgg16
from torchvision.models import VGG16_Weights

train_data_loader, val_data_loader = get_train_val_loaders()
paths = get_model_paths()

NUM_CLASSES = 3

if torch.backends.mps.is_available() and torch.backends.mps.is_built():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda:0")
else:
    DEVICE = torch.device("cpu")

# bbox 학습 안정화를 위해 backbone만 pretrained 사용
model = ssd300_vgg16(
    weights=None,
    weights_backbone=VGG16_Weights.DEFAULT,
    num_classes=NUM_CLASSES,
).to(DEVICE)

optim = torch.optim.SGD(
    model.parameters(),
    lr=1e-4,
    momentum=0.9,
    weight_decay=5e-4,
)

scheduler = torch.optim.lr_scheduler.StepLR(optim, step_size=2, gamma=0.1)
print("DEVICE:", DEVICE)
print("SAVE PATH:", paths["basic_pth"])

DEVICE: cpu
SAVE PATH: D:\00_repo\asdf\models\model.pth


In [ ]:
def _to_device_batch(images, targets, device):
    images = [img.to(device) for img in images]
    new_targets = []
    for t in targets:
        # SSD가 실제로 쓰는 키만 넘김
        new_targets.append({
            "boxes": t["boxes"].to(device),
            "labels": t["labels"].to(device),
        })
    return images, new_targets


def train_one_epoch(model, loader, optimizer, device, epoch, num_epochs):
    model.train()
    running = 0.0

    pbar = tqdm(loader, desc=f"[Train] {epoch}/{num_epochs}", leave=False)
    for images, targets in pbar:
        images, targets = _to_device_batch(images, targets, device)

        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())

        if not torch.isfinite(loss):
            continue

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        loss_val = float(loss.item())
        running += loss_val
        pbar.set_postfix(loss=f"{loss_val:.4f}")

    return running / max(len(loader), 1)


@torch.no_grad()
def validate_loss(model, loader, device, epoch, num_epochs):
    # detection 모델은 loss 계산 시 train 모드 필요
    model.train()
    running = 0.0

    pbar = tqdm(loader, desc=f"[Val] {epoch}/{num_epochs}", leave=False)
    for images, targets in pbar:
        images, targets = _to_device_batch(images, targets, device)

        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())
        loss_val = float(loss.item())
        running += loss_val
        pbar.set_postfix(loss=f"{loss_val:.4f}")

    return running / max(len(loader), 1)


num_epochs = 1
best_val = float("inf")

for epoch in range(1, num_epochs + 1):
    train_loss = train_one_epoch(model, train_data_loader, optim, DEVICE, epoch, num_epochs)
    val_loss = validate_loss(model, val_data_loader, DEVICE, epoch, num_epochs)
    scheduler.step()

    print(
        f"[Epoch {epoch}/{num_epochs}] "
        f"train_loss={train_loss:.4f} val_loss={val_loss:.4f} lr={optim.param_groups[0]['lr']:.2e}"
    )

    if val_loss < best_val:
        best_val = val_loss
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "num_classes": NUM_CLASSES,
                "best_val_loss": best_val,
            },
            paths["basic_pth"],
        )
        print(f"  -> best model saved: {paths['basic_pth']} (val_loss={best_val:.4f})")

In [8]:
model_path = paths["basic_pth"]
checkpoint = torch.load(model_path, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

SSD(
  (backbone): SSDFeatureExtractorVGG(
    (features): Sequential(
      (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU(inplace=True)
      (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU(inplace=True)
      (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (6): ReLU(inplace=True)
      (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (8): ReLU(inplace=True)
      (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (11): ReLU(inplace=True)
      (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (13): ReLU(inplace=True)
      (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (15): ReLU(inplace=

In [9]:
import copy

quantized_model = torch.ao.quantization.quantize_dynamic(
    copy.deepcopy(model),
    {torch.nn.Linear},
    dtype=torch.qint8,
)

torch.save(
    {
        "model_state_dict": quantized_model.state_dict(),
        "num_classes": NUM_CLASSES,
    },
    paths["quant_pth"],
)

C:\Users\pixar\AppData\Local\Temp\ipykernel_21184\2099098564.py:3: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.ao.quantization.quantize_dynamic(


In [10]:
# ONNX 변환
torch.onnx.export(
    copy.deepcopy(model),
    ([torch.randn(3, 300, 300, device=DEVICE)],),  # 모델 추적(tracing)에 사용할 더미 입력 이미지
    paths["onnx"],  # 저장될 ONNX 파일 이름
    export_params=True,
    dynamo=False,
    input_names=["images"],
    output_names=["boxes", "labels", "scores"],
    opset_version=17,
    do_constant_folding=True,
)

C:\Users\pixar\AppData\Local\Temp\ipykernel_21184\1212220179.py:2: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
d:\00_repo\asdf\.venv\Lib\site-packages\torchvision\ops\boxes.py:174: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  boxes_x = torch.min(boxes_x, torch.tensor(width, dtype=boxes.dtype, device=boxes.device))
d:\00_repo\asdf\.venv\Lib\site-packages\torchvision\ops\boxes.py:176: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or 